# Concentric-Tube CHT Analysis

## Predictive physical model

This notebook develops the preliminary analysis for a concentric-tube
conjugate heat-transfer problem.

### Primary interpretation

- The hot-gas inlet represents a reservoir at 10 bar absolute and 1000 K.
- The gas discharges into atmospheric pressure.
- The gas mass flow rate is predicted rather than prescribed.
- Liquid water enters the annulus at 15 bar absolute and 300 K.
- The coolant mass flow rate is 0.5 kg/s.
- The gas and coolant flow in opposite directions.
- The coolant outlet pressure is calculated from the pressure loss.
- The external gas jet is outside the analytical domain.

All calculations use SI units.

In [60]:
# Geometry [m]
length = 20e-3

inner_tube_id = 10e-3
inner_tube_od = 12e-3

outer_tube_id = 14e-3
outer_tube_od = 16e-3

# Hot-gas reservoir conditions
gas_total_temperature = 1000.0       # K
gas_total_pressure = 10.0e5          # Pa absolute
back_pressure = 101325.0             # Pa absolute

# Gas composition: mole fractions
gas_mole_fractions = {
    "H2O": 0.5,
    "CO2": 0.5,
}

# Coolant inlet conditions
coolant_inlet_temperature = 300.0    # K
coolant_inlet_pressure = 15.0e5      # Pa absolute
coolant_mass_flow_rate = 0.5         # kg/s

# Baseline copper property
copper_conductivity = 387.6          # W/(m K)

## 1. Geometry

The hot gas flows through the circular passage inside the inner copper tube.

The liquid-water coolant flows through the annulus between the inner tube
outer diameter and the outer tube inner diameter.

For a concentric annulus, the hydraulic diameter is

$$
D_{h,c}
=
\frac{4A_c}{P_{\mathrm{wet},c}}
=
D_{o,c}-D_{i,c}
$$

The gas-side and coolant-side heat-transfer areas differ because they lie at
different radii.

In [61]:
import numpy as np

# Gas flow area
gas_flow_area = np.pi * inner_tube_id**2 / 4

# Coolant annular flow area
coolant_flow_area = (
    np.pi
    * (outer_tube_id**2 - inner_tube_od**2)
    / 4
)

# Wetted perimeters
gas_wetted_perimeter = np.pi * inner_tube_id

coolant_wetted_perimeter = np.pi * (
    outer_tube_id + inner_tube_od
)

# Hydraulic diameters
gas_hydraulic_diameter = inner_tube_id

coolant_hydraulic_diameter = (
    4 * coolant_flow_area / coolant_wetted_perimeter
)

# Heat-transfer areas
gas_heat_transfer_area = (
    np.pi * inner_tube_id * length
)

coolant_heat_transfer_area = (
    np.pi * inner_tube_od * length
)

# Copper wall cross-sectional area
inner_copper_cross_section = (
    np.pi
    * (inner_tube_od**2 - inner_tube_id**2)
    / 4
)

# Passage aspect ratios
gas_length_ratio = length / gas_hydraulic_diameter
coolant_length_ratio = length / coolant_hydraulic_diameter

In [62]:
print("GEOMETRY SUMMARY")
print("-" * 45)

print(f"Gas flow area:              {gas_flow_area:.6e} m²")
print(f"Coolant flow area:          {coolant_flow_area:.6e} m²")

print(
    f"Gas hydraulic diameter:     "
    f"{gas_hydraulic_diameter * 1e3:.3f} mm"
)

print(
    f"Coolant hydraulic diameter: "
    f"{coolant_hydraulic_diameter * 1e3:.3f} mm"
)

print(
    f"Gas heat-transfer area:     "
    f"{gas_heat_transfer_area:.6e} m²"
)

print(
    f"Coolant heat-transfer area: "
    f"{coolant_heat_transfer_area:.6e} m²"
)

print(f"Gas L/Dh:                   {gas_length_ratio:.2f}")
print(f"Coolant L/Dh:               {coolant_length_ratio:.2f}")

GEOMETRY SUMMARY
---------------------------------------------
Gas flow area:              7.853982e-05 m²
Coolant flow area:          4.084070e-05 m²
Gas hydraulic diameter:     10.000 mm
Coolant hydraulic diameter: 2.000 mm
Gas heat-transfer area:     6.283185e-04 m²
Coolant heat-transfer area: 7.539822e-04 m²
Gas L/Dh:                   2.00
Coolant L/Dh:               10.00


## 2. Coolant properties

The coolant is initially modelled as single-phase liquid water.

Its thermophysical properties are evaluated at the inlet static state:

$$
T_{c,\mathrm{in}} = 300\ \mathrm{K}
$$

$$
p_{c,\mathrm{in}} = 15\ \mathrm{bar(a)}
$$

The local static pressure is used because water properties and saturation
temperature depend on the thermodynamic pressure, not the gauge pressure.

In [63]:
from CoolProp.CoolProp import PropsSI

# Coolant properties at the inlet state
coolant_density = PropsSI(
    "D",
    "T", coolant_inlet_temperature,
    "P", coolant_inlet_pressure,
    "Water",
)

coolant_cp = PropsSI(
    "C",
    "T", coolant_inlet_temperature,
    "P", coolant_inlet_pressure,
    "Water",
)

coolant_viscosity = PropsSI(
    "V",
    "T", coolant_inlet_temperature,
    "P", coolant_inlet_pressure,
    "Water",
)

coolant_conductivity = PropsSI(
    "L",
    "T", coolant_inlet_temperature,
    "P", coolant_inlet_pressure,
    "Water",
)

coolant_prandtl = (
    coolant_cp
    * coolant_viscosity
    / coolant_conductivity
)

# Saturation temperature at the inlet pressure
coolant_saturation_temperature = PropsSI(
    "T",
    "P", coolant_inlet_pressure,
    "Q", 0,
    "Water",
)

In [64]:
print("COOLANT INLET PROPERTIES")
print("-" * 45)

print(f"Density:                  {coolant_density:.3f} kg/m³")
print(f"Specific heat:            {coolant_cp:.2f} J/(kg K)")
print(f"Dynamic viscosity:        {coolant_viscosity:.6e} Pa s")
print(f"Thermal conductivity:     {coolant_conductivity:.4f} W/(m K)")
print(f"Prandtl number:           {coolant_prandtl:.3f}")

print(
    f"Saturation temperature:   "
    f"{coolant_saturation_temperature:.2f} K"
)

print(
    f"Inlet bulk subcooling:    "
    f"{coolant_saturation_temperature - coolant_inlet_temperature:.2f} K"
)

COOLANT INLET PROPERTIES
---------------------------------------------
Density:                  997.184 kg/m³
Specific heat:            4176.70 J/(kg K)
Dynamic viscosity:        8.536188e-04 Pa s
Thermal conductivity:     0.6103 W/(m K)
Prandtl number:           5.842
Saturation temperature:   471.44 K
Inlet bulk subcooling:    171.44 K


## 3. Coolant flow quantities

The coolant mean velocity follows from conservation of mass:

$$
\dot{m}_c = \rho_c u_c A_c
$$

Therefore,

$$
u_c =
\frac{\dot{m}_c}{\rho_c A_c}
$$

The coolant mass flux is

$$
G_c = \frac{\dot{m}_c}{A_c}
$$

The Reynolds number is

$$
Re_c =
\frac{\rho_c u_c D_{h,c}}{\mu_c}
$$

and determines whether the coolant flow is laminar, transitional or turbulent.

The Péclet number is

$$
Pe_c = Re_c Pr_c
$$

and compares axial advection of thermal energy with thermal diffusion.

In [65]:
# Coolant mean velocity
coolant_velocity = (
    coolant_mass_flow_rate
    / (coolant_density * coolant_flow_area)
)

# Coolant mass flux
coolant_mass_flux = (
    coolant_mass_flow_rate
    / coolant_flow_area
)

# Reynolds number
coolant_reynolds = (
    coolant_density
    * coolant_velocity
    * coolant_hydraulic_diameter
    / coolant_viscosity
)

# Péclet number
coolant_peclet = (
    coolant_reynolds
    * coolant_prandtl
)

# Dynamic pressure
coolant_dynamic_pressure = (
    0.5
    * coolant_density
    * coolant_velocity**2
)

# Classify the flow regime
if coolant_reynolds < 2300:
    coolant_flow_regime = "laminar"
elif coolant_reynolds < 4000:
    coolant_flow_regime = "transitional"
else:
    coolant_flow_regime = "turbulent"

In [66]:
print("COOLANT FLOW SUMMARY")
print("-" * 45)

print(
    f"Mean velocity:             "
    f"{coolant_velocity:.3f} m/s"
)

print(
    f"Mass flux:                 "
    f"{coolant_mass_flux:.1f} kg/(m² s)"
)

print(
    f"Reynolds number:           "
    f"{coolant_reynolds:.3e}"
)

print(
    f"Prandtl number:            "
    f"{coolant_prandtl:.3f}"
)

print(
    f"Péclet number:             "
    f"{coolant_peclet:.3e}"
)

print(
    f"Dynamic pressure:          "
    f"{coolant_dynamic_pressure / 1e5:.3f} bar"
)

print(
    f"Flow regime:               "
    f"{coolant_flow_regime}"
)

COOLANT FLOW SUMMARY
---------------------------------------------
Mean velocity:             12.277 m/s
Mass flux:                 12242.7 kg/(m² s)
Reynolds number:           2.868e+04
Prandtl number:            5.842
Péclet number:             1.676e+05
Dynamic pressure:          0.752 bar
Flow regime:               turbulent


## 4. Preliminary coolant pressure drop

The smooth-wall Darcy friction factor is estimated with the
Petukhov–Filonenko correlation:

$$
f_c =
\left(0.79\ln Re_c-1.64\right)^{-2}
$$

The distributed frictional pressure drop is then calculated using the
Darcy–Weisbach equation:

$$
\Delta p_{\mathrm{fric},c}
=
f_c
\frac{L}{D_{h,c}}
\frac{\rho_c u_c^2}{2}
$$

Possible entrance, exit and geometric losses are represented separately by:

$$
\Delta p_{\mathrm{minor},c}
=
K_{\mathrm{total}}
\frac{\rho_c u_c^2}{2}
$$

The geometry of the connections is not provided, so no particular minor-loss
coefficient is assumed at this stage.

In [67]:
# Check that the turbulent friction correlation is applicable
if coolant_reynolds <= 3000:
    raise ValueError(
        "Petukhov–Filonenko requires turbulent flow."
    )

# Smooth-wall Darcy friction factor
coolant_friction_factor = (
    0.79 * np.log(coolant_reynolds) - 1.64
) ** -2

# Distributed wall-friction pressure drop
coolant_friction_pressure_drop = (
    coolant_friction_factor
    * (length / coolant_hydraulic_diameter)
    * coolant_dynamic_pressure
)

# Pressure-loss scale for a minor-loss coefficient K = 1
coolant_minor_loss_per_K = coolant_dynamic_pressure

# Preliminary outlet pressure including distributed friction only
coolant_outlet_pressure_friction_only = (
    coolant_inlet_pressure
    - coolant_friction_pressure_drop
)

In [68]:
print("COOLANT PRESSURE-DROP SUMMARY")
print("-" * 50)

print(
    f"Darcy friction factor:          "
    f"{coolant_friction_factor:.5f}"
)

print(
    f"Length-to-diameter ratio:       "
    f"{length / coolant_hydraulic_diameter:.2f}"
)

print(
    f"Distributed pressure drop:      "
    f"{coolant_friction_pressure_drop / 1e5:.4f} bar"
)

print(
    f"Minor-loss pressure per K = 1:  "
    f"{coolant_minor_loss_per_K / 1e5:.4f} bar"
)

print(
    f"Friction-only outlet pressure:  "
    f"{coolant_outlet_pressure_friction_only / 1e5:.4f} bar(a)"
)

COOLANT PRESSURE-DROP SUMMARY
--------------------------------------------------
Darcy friction factor:          0.02390
Length-to-diameter ratio:       10.00
Distributed pressure drop:      0.1796 bar
Minor-loss pressure per K = 1:  0.7515 bar
Friction-only outlet pressure:  14.8204 bar(a)


## 5. Coolant hydrodynamic development

A turbulent flow entering a passage requires a finite distance for its velocity
profile and wall shear stress to become fully developed.

Two approximate turbulent entrance-length estimates are considered:

$$
L_{e,1} \approx 10D_h
$$

and

$$
\frac{L_{e,2}}{D_h}
\approx
4.4Re^{1/6}
$$

These estimates are screening relations rather than exact annulus-specific
correlations.

The actual development length depends on the upstream velocity profile,
inlet geometry, turbulence level and annular radius ratio.

In [69]:
# Simple turbulent entrance-length estimate
coolant_entrance_length_10Dh = (
    10 * coolant_hydraulic_diameter
)

# Reynolds-number-dependent turbulent estimate
coolant_entrance_length_empirical = (
    4.4
    * coolant_reynolds**(1 / 6)
    * coolant_hydraulic_diameter
)

# Fraction of each estimated entrance length covered by the tube
development_fraction_10Dh = (
    length / coolant_entrance_length_10Dh
)

development_fraction_empirical = (
    length / coolant_entrance_length_empirical
)

In [70]:
print("COOLANT DEVELOPMENT-LENGTH SUMMARY")
print("-" * 55)

print(
    f"Actual passage length:              "
    f"{length * 1e3:.2f} mm"
)

print(
    f"Estimate using 10 Dh:               "
    f"{coolant_entrance_length_10Dh * 1e3:.2f} mm"
)

print(
    f"Estimate using 4.4 Re^(1/6) Dh:     "
    f"{coolant_entrance_length_empirical * 1e3:.2f} mm"
)

print(
    f"L / Le using 10 Dh:                 "
    f"{development_fraction_10Dh:.3f}"
)

print(
    f"L / Le using empirical estimate:    "
    f"{development_fraction_empirical:.3f}"
)

COOLANT DEVELOPMENT-LENGTH SUMMARY
-------------------------------------------------------
Actual passage length:              20.00 mm
Estimate using 10 Dh:               20.00 mm
Estimate using 4.4 Re^(1/6) Dh:     48.69 mm
L / Le using 10 Dh:                 1.000
L / Le using empirical estimate:    0.411


The coolant pressure-drop calculation using fully developed friction provides a baseline. Developing-flow and entrance losses remain an uncertainty because the upstream geometry is unspecified.

## Coolant Convection and Copper-Wall Conduction

### Coolant-side convection

The fully developed turbulent Nusselt number is estimated using the Gnielinski correlation:

$$
Nu_c =
\frac{(f_D/8)(Re_c-1000)Pr_c}
{1+12.7(f_D/8)^{1/2}(Pr_c^{2/3}-1)}
$$

The coolant-side heat-transfer coefficient is:

$$
h_c=\frac{Nu_c k_c}{D_h}
$$

This is used as a baseline estimate; entrance and annulus effects will be considered separately.

### Copper-wall conduction

The radial conduction resistance through the inner copper tube is:

$$
R_{\mathrm{Cu}}
=
\frac{\ln(D_o/D_i)}
{2\pi k_{\mathrm{Cu}}L}
$$

The gas-side and coolant-side heat-transfer areas are:

$$
A_g=\pi D_iL,
\qquad
A_c=\pi D_oL
$$

Since the two areas differ, the same heat-transfer rate produces different surface heat fluxes:

$$
q_g''=\frac{Q}{A_g},
\qquad
q_c''=\frac{Q}{A_c}
$$

In [71]:
# Coolant-side heat-transfer coefficient:
# Fully developed turbulent baseline using Gnielinski correlation

# Darcy friction factor from the previous step
f_D = coolant_friction_factor

# Gnielinski correlation
coolant_nusselt = ((f_D / 8.0) * (coolant_reynolds - 1000.0) * coolant_prandtl) / (
    1.0 + 12.7 * (f_D / 8.0)**0.5 * (coolant_prandtl**(2.0 / 3.0) - 1.0)
)

# Convective heat-transfer coefficient
coolant_ht_coefficient = coolant_nusselt * coolant_conductivity / coolant_hydraulic_diameter

print(f"Fully developed Nusselt number:      {coolant_nusselt:.1f}")
print(f"Coolant heat-transfer coefficient:   {coolant_ht_coefficient:.0f} W/(m²·K)")

Fully developed Nusselt number:      188.9
Coolant heat-transfer coefficient:   57647 W/(m²·K)


In [72]:
import numpy as np

# Inner copper tube geometry
D_gas_wall = 10e-3       # Gas-side diameter [m]
D_coolant_wall = 12e-3   # Coolant-side diameter [m]
L = 20e-3                # Heated length [m]

# Copper thermal conductivity
k_copper = 380.0         # W/(m·K), initial constant-property value

# Heat-transfer areas
A_gas_wall = np.pi * D_gas_wall * L
A_coolant_wall = np.pi * D_coolant_wall * L

# Cylindrical radial-conduction resistance
R_copper = np.log(D_coolant_wall / D_gas_wall) / (
    2.0 * np.pi * k_copper * L
)

print(f"Gas-side heat-transfer area:       {A_gas_wall:.6e} m²")
print(f"Coolant-side heat-transfer area:   {A_coolant_wall:.6e} m²")
print(f"Copper conduction resistance:      {R_copper:.6e} K/W")

Gas-side heat-transfer area:       6.283185e-04 m²
Coolant-side heat-transfer area:   7.539822e-04 m²
Copper conduction resistance:      3.818076e-03 K/W


In [73]:
# Coolant-side convective thermal resistance
R_coolant = 1.0 / (coolant_ht_coefficient * A_coolant_wall)

# Comparison between copper conduction and coolant convection
resistance_ratio = R_copper / R_coolant

# Combined resistance currently modelled
R_partial = R_copper + R_coolant

copper_fraction = 100.0 * R_copper / R_partial
coolant_fraction = 100.0 * R_coolant / R_partial

print(f"Coolant convection resistance:  {R_coolant:.6e} K/W")
print(f"Copper conduction resistance:   {R_copper:.6e} K/W")
print(f"R_copper / R_coolant:           {resistance_ratio:.4f}")
print()
print(f"Copper contribution:            {copper_fraction:.1f} %")
print(f"Coolant contribution:           {coolant_fraction:.1f} %")

Coolant convection resistance:  2.300718e-02 K/W
Copper conduction resistance:   3.818076e-03 K/W
R_copper / R_coolant:           0.1660

Copper contribution:            14.2 %
Coolant contribution:           85.8 %


In [74]:
import cantera as ct

print(f"Cantera version: {ct.__version__}")

Cantera version: 3.2.0


## Gas-Mixture Properties

The gas is modelled as an equimolar mixture of steam and carbon dioxide:

$$
X_{\mathrm{H_2O}}=X_{\mathrm{CO_2}}=0.5
$$

The specified conditions are interpreted as upstream stagnation conditions:

$$
T_0=1000\ \mathrm{K},
\qquad
p_0=10\ \mathrm{bar(a)}
$$

Temperature-dependent mixture properties are evaluated using Cantera with the `gri30.yaml` thermodynamic and transport database. No chemical equilibrium or reaction calculation is performed.

The mixture Prandtl number and specific-heat ratio are calculated as:

$$
Pr=\frac{c_p\mu}{k},
\qquad
\gamma=\frac{c_p}{c_v}
$$


These are reservoir reference properties; local static properties inside the tube will change as the gas accelerates and cools.

In [75]:

# Gas inlet conditions
T_gas_in = 1000.0       # K
p_gas_in = 10.0e5       # Pa absolute

# Create gas mixture using the GRI-Mech species database
gas = ct.Solution("gri30.yaml")

# Equimolar mixture: mole fractions of 0.5 H2O and 0.5 CO2
gas.TPX = T_gas_in, p_gas_in, {"H2O": 0.5, "CO2": 0.5}

# Mixture properties
rho_gas = gas.density
cp_gas = gas.cp_mass
cv_gas = gas.cv_mass
gamma_gas = cp_gas / cv_gas
mu_gas = gas.viscosity
k_gas = gas.thermal_conductivity
Pr_gas = cp_gas * mu_gas / k_gas
M_gas = gas.mean_molecular_weight
R_gas = ct.gas_constant / M_gas
a_gas = gas.sound_speed

print(f"Mixture molecular weight:       {M_gas:.3f} kg/kmol")
print(f"Specific gas constant:          {R_gas:.2f} J/(kg·K)")
print(f"Density:                        {rho_gas:.4f} kg/m³")
print(f"Specific heat cp:               {cp_gas:.1f} J/(kg·K)")
print(f"Specific heat ratio gamma:      {gamma_gas:.4f}")
print(f"Dynamic viscosity:              {mu_gas:.4e} Pa·s")
print(f"Thermal conductivity:           {k_gas:.4f} W/(m·K)")
print(f"Prandtl number:                 {Pr_gas:.3f}")
print(f"Speed of sound:                 {a_gas:.1f} m/s")

Mixture molecular weight:       31.012 kg/kmol
Specific gas constant:          268.10 J/(kg·K)
Density:                        3.7299 kg/m³
Specific heat cp:               1541.6 J/(kg·K)
Specific heat ratio gamma:      1.2105
Dynamic viscosity:              3.9847e-05 Pa·s
Thermal conductivity:           0.0899 W/(m·K)
Prandtl number:                 0.683
Speed of sound:                 569.7 m/s


## Ideal Choked-Flow Screening

The gas discharges from upstream stagnation conditions $(p_0,T_0)$ to atmospheric back pressure $p_b$.

Choking is expected when:

$$
\frac{p_b}{p_0}
\leq
\left(\frac{2}{\gamma+1}\right)^{\gamma/(\gamma-1)}
$$

The corresponding critical pressure and temperature are:

$$
p^*=p_0
\left(\frac{2}{\gamma+1}\right)^{\gamma/(\gamma-1)}
$$

$$
T^*=T_0\left(\frac{2}{\gamma+1}\right)
$$

The ideal choked mass-flow rate is:

$$
\dot m_{\mathrm{ideal}}
=
A p_0\sqrt{\frac{\gamma}{RT_0}}
\left(\frac{2}{\gamma+1}\right)^{
\frac{\gamma+1}{2(\gamma-1)}
}
$$

This provides a screening estimate and upper reference. Tube friction, heat transfer and entrance losses are not yet included.

In [76]:
import numpy as np
import cantera as ct

# Gas-tube geometry
D_gas = 10.0e-3
A_gas_flow = np.pi * D_gas**2 / 4.0

# Back pressure: atmospheric absolute pressure
p_back = ct.one_atm

# Critical pressure ratio for an ideal gas
critical_pressure_ratio = (
    2.0 / (gamma_gas + 1.0)
)**(gamma_gas / (gamma_gas - 1.0))

p_critical = p_gas_in * critical_pressure_ratio
T_critical = T_gas_in * 2.0 / (gamma_gas + 1.0)

# Ideal choked mass-flow rate, assuming Cd = 1
mass_flux_choked = (
    p_gas_in
    * np.sqrt(gamma_gas / (R_gas * T_gas_in))
    * (2.0 / (gamma_gas + 1.0))
    ** ((gamma_gas + 1.0) / (2.0 * (gamma_gas - 1.0)))
)

m_dot_choked_ideal = mass_flux_choked * A_gas_flow

print(f"Back-pressure ratio p_back/p0:  {p_back/p_gas_in:.4f}")
print(f"Critical pressure ratio:        {critical_pressure_ratio:.4f}")
print(f"Critical pressure:              {p_critical/1e5:.3f} bar(a)")
print(f"Critical temperature:           {T_critical:.1f} K")
print(f"Ideal choked mass flux:         {mass_flux_choked:.1f} kg/(m²·s)")
print(f"Ideal choked mass-flow rate:    {m_dot_choked_ideal:.5f} kg/s")

Back-pressure ratio p_back/p0:  0.1013
Critical pressure ratio:        0.5624
Critical pressure:              5.624 bar(a)
Critical temperature:           904.8 K
Ideal choked mass flux:         1256.4 kg/(m²·s)
Ideal choked mass-flow rate:    0.09868 kg/s


## Gas-side reference flow quantities

The ideal choked mass flow is used provisionally to estimate the gas-side flow
regime:

$$
u_{\mathrm{ref}}=\frac{\dot{m}_g}{\rho_g A_g},
\qquad
Re_{\mathrm{ref}}=\frac{\rho_g u_{\mathrm{ref}}D_i}{\mu_g},
\qquad
Ma_{\mathrm{ref}}=\frac{u_{\mathrm{ref}}}{a_g}.
$$

These are reference values based on properties at \(10\) bar and \(1000\) K.
They are not the actual inlet or exit conditions because gas density changes
significantly through the tube.

In [77]:
# Gas-tube geometry
D_gas = 0.010  # m
A_gas = np.pi * D_gas**2 / 4.0

# Provisional gas mass flow from ideal choking calculation
mdot_gas_ref = m_dot_choked_ideal

# Reference flow quantities at 10 bar and 1000 K
u_gas_ref = mdot_gas_ref / (rho_gas * A_gas)
Re_gas_ref = rho_gas * u_gas_ref * D_gas / mu_gas
Ma_gas_ref = u_gas_ref / a_gas

print(f"Gas flow area:              {A_gas:.6e} m²")
print(f"Reference mass flow:        {mdot_gas_ref:.5f} kg/s")
print(f"Reference bulk velocity:    {u_gas_ref:.1f} m/s")
print(f"Reference Reynolds number:  {Re_gas_ref:.3e}")
print(f"Reference Mach number:      {Ma_gas_ref:.3f}")

Gas flow area:              7.853982e-05 m²
Reference mass flow:        0.09868 kg/s
Reference bulk velocity:    336.9 m/s
Reference Reynolds number:  3.153e+05
Reference Mach number:      0.591


## Baseline gas-side heat-transfer coefficient

Since the reference Reynolds number is turbulent, the Gnielinski correlation
is used as baseline.

The Darcy friction factor for a smooth tube is

$$
f = \left(0.79\ln Re_g-1.64\right)^{-2}.
$$

The Nusselt number is

$$
Nu_g =
\frac{(f/8)(Re_g-1000)Pr_g}
{1+12.7(f/8)^{1/2}\left(Pr_g^{2/3}-1\right)}.
$$

The corresponding gas-side heat-transfer coefficient is

$$
h_g = \frac{Nu_g k_g}{D_i}.
$$

This correlation assumes fully developed turbulent flow. Since the tube has
$L/D_i=2$, the result is retained only as a baseline; entrance and compressibility
effects are not included in Model 0.

In [78]:
# Gas-side baseline: fully developed turbulent flow

L_tube = 0.020  # m

f_gas = (0.79 * np.log(Re_gas_ref) - 1.64) ** -2

Nu_gas = (
    (f_gas / 8.0) * (Re_gas_ref - 1000.0) * Pr_gas
    / (
        1.0
        + 12.7
        * np.sqrt(f_gas / 8.0)
        * (Pr_gas ** (2.0 / 3.0) - 1.0)
    )
)

h_gas = Nu_gas * k_gas / D_gas
L_over_D_gas = L_tube / D_gas

print(f"Darcy friction factor:       {f_gas:.5f}")
print(f"Gas-side Nusselt number:     {Nu_gas:.1f}")
print(f"Gas-side HTC:                {h_gas:.1f} W/(m²·K)")
print(f"Tube length ratio L/D:       {L_over_D_gas:.1f}")

Darcy friction factor:       0.01430
Gas-side Nusselt number:     436.3
Gas-side HTC:                3924.1 W/(m²·K)
Tube length ratio L/D:       2.0


## Radial thermal-resistance network

Heat passes from the hot gas through three resistances in series:

$$
R_g=\frac{1}{h_g\pi D_iL}
$$

$$
R_{\mathrm{Cu}}=
\frac{\ln(D_o/D_i)}
{2\pi k_{\mathrm{Cu}}L}
$$

$$
R_c=\frac{1}{h_c\pi D_oL}
$$

Therefore,

$$
R_{\mathrm{total}}=R_g+R_{\mathrm{Cu}}+R_c
$$

and

$$
UA=\frac{1}{R_{\mathrm{total}}}.
$$

In [79]:
# Thermal resistances
R_gas = 1.0 / (h_gas * A_gas_wall)

R_total = R_gas + R_copper + R_coolant
UA = 1.0 / R_total

print(f"Gas convection resistance:      {R_gas:.5f} K/W")
print(f"Copper conduction resistance:   {R_copper:.5f} K/W")
print(f"Coolant convection resistance:  {R_coolant:.5f} K/W")
print(f"Total thermal resistance:       {R_total:.5f} K/W")
print(f"Overall conductance UA:         {UA:.3f} W/K")

print("\nResistance contributions")
print(f"Gas convection:      {100*R_gas/R_total:.2f} %")
print(f"Copper conduction:   {100*R_copper/R_total:.2f} %")
print(f"Coolant convection:  {100*R_coolant/R_total:.2f} %")

Gas convection resistance:      0.40558 K/W
Copper conduction resistance:   0.00382 K/W
Coolant convection resistance:  0.02301 K/W
Total thermal resistance:       0.43241 K/W
Overall conductance UA:         2.313 W/K

Resistance contributions
Gas convection:      93.80 %
Copper conduction:   0.88 %
Coolant convection:  5.32 %


## Heat-transfer rate

For the initial lumped model, the gas and coolant bulk temperatures are
approximated by their inlet values.

The heat-transfer rate is

$$
\dot Q=
\frac{T_{g,in}-T_{c,in}}
{R_{\mathrm{total}}}.
$$

The heat flux differs on the two wall surfaces because their areas differ:

$$
q''_g=\frac{\dot Q}{\pi D_iL},
\qquad
q''_c=\frac{\dot Q}{\pi D_oL}.
$$

Although the heat fluxes differ, conservation of energy requires the same
heat-transfer rate through every layer:

$$
\dot Q_g=\dot Q_{\mathrm{Cu}}=\dot Q_c.
$$

In [80]:
delta_T_bulk = gas_total_temperature - coolant_inlet_temperature

# Heat-transfer rate
Q_dot = delta_T_bulk / R_total

# Heat fluxes based on the two different wall areas
q_flux_gas = Q_dot / A_gas_wall
q_flux_coolant = Q_dot / A_coolant_wall

print(f"Bulk temperature difference:  {delta_T_bulk:.1f} K")
print(f"Heat-transfer rate:            {Q_dot:.1f} W")
print(f"Gas-side heat flux:            {q_flux_gas/1e6:.3f} MW/m²")
print(f"Coolant-side heat flux:        {q_flux_coolant/1e6:.3f} MW/m²")

Bulk temperature difference:  700.0 K
Heat-transfer rate:            1618.8 W
Gas-side heat flux:            2.576 MW/m²
Coolant-side heat flux:        2.147 MW/m²


## Wall and outlet temperatures

The temperature drop across each thermal resistance is

$$
\Delta T_i=\dot Q R_i.
$$

The wall temperatures are

$$
T_{w,g}=T_{g,\mathrm{bulk}}-\dot Q R_g
$$

and

$$
T_{w,c}=T_{c,\mathrm{bulk}}+\dot Q R_c.
$$

The copper conduction temperature drop must satisfy

$$
T_{w,g}-T_{w,c}=\dot Q R_{\mathrm{Cu}}.
$$

The bulk-fluid temperature changes are calculated from

$$
\Delta T_g=\frac{\dot Q}{\dot m_g c_{p,g}},
\qquad
\Delta T_c=\frac{\dot Q}{\dot m_c c_{p,c}}.
$$

In [81]:
# Temperature drops across individual resistances
delta_T_gas_film = Q_dot * R_gas
delta_T_copper = Q_dot * R_copper
delta_T_coolant_film = Q_dot * R_coolant

# Copper-wall temperatures
T_wall_gas = gas_total_temperature - delta_T_gas_film
T_wall_coolant = coolant_inlet_temperature + delta_T_coolant_film

# Fluid bulk-temperature changes
delta_T_gas_bulk = Q_dot / (mdot_gas_ref * cp_gas)
delta_T_coolant_bulk = Q_dot / (coolant_mass_flow_rate * coolant_cp)

# Approximate outlet temperatures
T_gas_out = gas_total_temperature - delta_T_gas_bulk
T_coolant_out = coolant_inlet_temperature + delta_T_coolant_bulk

print("TEMPERATURE-DROP SUMMARY")
print("-" * 42)
print(f"Gas convection drop:          {delta_T_gas_film:.2f} K")
print(f"Copper conduction drop:       {delta_T_copper:.2f} K")
print(f"Coolant convection drop:      {delta_T_coolant_film:.2f} K")

print("\nWALL TEMPERATURES")
print("-" * 42)
print(f"Gas-side wall temperature:    {T_wall_gas:.2f} K")
print(f"Coolant-side wall temperature:{T_wall_coolant:.2f} K")
print(
    f"Wall difference check:        "
    f"{T_wall_gas - T_wall_coolant:.2f} K"
)

print("\nFLUID TEMPERATURE CHANGES")
print("-" * 42)
print(f"Gas temperature decrease:     {delta_T_gas_bulk:.2f} K")
print(f"Coolant temperature increase: {delta_T_coolant_bulk:.2f} K")
print(f"Approximate gas outlet:       {T_gas_out:.2f} K")
print(f"Approximate coolant outlet:   {T_coolant_out:.2f} K")

TEMPERATURE-DROP SUMMARY
------------------------------------------
Gas convection drop:          656.57 K
Copper conduction drop:       6.18 K
Coolant convection drop:      37.24 K

WALL TEMPERATURES
------------------------------------------
Gas-side wall temperature:    343.43 K
Coolant-side wall temperature:337.24 K
Wall difference check:        6.18 K

FLUID TEMPERATURE CHANGES
------------------------------------------
Gas temperature decrease:     10.64 K
Coolant temperature increase: 0.78 K
Approximate gas outlet:       989.36 K
Approximate coolant outlet:   300.78 K


## Energy-conservation checks

At steady state, the same heat-transfer rate must pass through every layer:

$$
\dot Q_g=\dot Q_{\mathrm{Cu}}=\dot Q_c.
$$

The resistance-network heat rates are

$$
\dot Q_g=\frac{T_{g,\mathrm{bulk}}-T_{w,g}}{R_g},
$$

$$
\dot Q_{\mathrm{Cu}}=
\frac{T_{w,g}-T_{w,c}}{R_{\mathrm{Cu}}},
$$

$$
\dot Q_c=
\frac{T_{w,c}-T_{c,\mathrm{bulk}}}{R_c}.
$$

The fluid energy balances must also satisfy

$$
\dot Q=\dot m_g c_{p,g}(T_{g,in}-T_{g,out})
$$

and

$$
\dot Q=\dot m_c c_{p,c}(T_{c,out}-T_{c,in}).
$$

In [82]:
# Heat rates recovered from individual thermal resistances
Q_from_gas_film = (
    (gas_total_temperature - T_wall_gas) / R_gas
)

Q_from_copper = (
    (T_wall_gas - T_wall_coolant) / R_copper
)

Q_from_coolant_film = (
    (T_wall_coolant - coolant_inlet_temperature) / R_coolant
)

# Heat rates recovered from fluid energy balances
Q_from_gas_energy = (
    mdot_gas_ref * cp_gas * (gas_total_temperature - T_gas_out)
)

Q_from_coolant_energy = (
    coolant_mass_flow_rate * coolant_cp
    * (T_coolant_out - coolant_inlet_temperature)
)

heat_rates = np.array([
    Q_dot,
    Q_from_gas_film,
    Q_from_copper,
    Q_from_coolant_film,
    Q_from_gas_energy,
    Q_from_coolant_energy
])

relative_spread = (
    100.0 * (heat_rates.max() - heat_rates.min()) / Q_dot
)

print("ENERGY-CONSERVATION CHECK")
print("-" * 44)
print(f"Resistance-network heat rate: {Q_dot:.3f} W")
print(f"From gas convection:          {Q_from_gas_film:.3f} W")
print(f"Through copper:               {Q_from_copper:.3f} W")
print(f"To coolant:                   {Q_from_coolant_film:.3f} W")
print(f"From gas energy balance:      {Q_from_gas_energy:.3f} W")
print(f"From coolant energy balance:  {Q_from_coolant_energy:.3f} W")
print(f"Maximum relative spread:      {relative_spread:.3e} %")

# Basic physical checks
assert gas_total_temperature > T_wall_gas
assert T_wall_gas > T_wall_coolant
assert T_wall_coolant > coolant_inlet_temperature
assert relative_spread < 1e-8

print("\nAll Model 0 energy checks passed.")

ENERGY-CONSERVATION CHECK
--------------------------------------------
Resistance-network heat rate: 1618.842 W
From gas convection:          1618.842 W
Through copper:               1618.842 W
To coolant:                   1618.842 W
From gas energy balance:      1618.842 W
From coolant energy balance:  1618.842 W
Maximum relative spread:      3.694e-12 %

All Model 0 energy checks passed.


## Coolant boiling screening

The saturation temperature of water is evaluated at the assumed coolant
pressure:

$$
T_{\mathrm{sat}}=T_{\mathrm{sat}}(p_c).
$$

The bulk-liquid subcooling is

$$
\Delta T_{\mathrm{sub,bulk}}
=
T_{\mathrm{sat}}-T_{c,\mathrm{bulk}},
$$

and the wall subcooling is

$$
\Delta T_{\mathrm{sub,wall}}
=
T_{\mathrm{sat}}-T_{w,c}.
$$

If $T_{w,c}<T_{\mathrm{sat}}$, the wall remains below the saturation
temperature and boiling is not predicted by Model 0.

In [83]:
# Saturation temperature at coolant pressure
p_coolant = 15*10**5 # Pa(a)
T_sat_coolant = coolant_saturation_temperature
# Subcooling margins
bulk_subcooling = T_sat_coolant - coolant_inlet_temperature
wall_subcooling = T_sat_coolant - T_wall_coolant
outlet_subcooling = T_sat_coolant - T_coolant_out

boiling_predicted = T_wall_coolant >= T_sat_coolant

print("COOLANT BOILING SCREENING")
print("-" * 43)
print(f"Coolant pressure:             {p_coolant/1e5:.2f} bar(a)")
print(f"Saturation temperature:       {T_sat_coolant:.2f} K")
print(f"Coolant bulk temperature:     {coolant_inlet_temperature:.2f} K")
print(f"Coolant outlet temperature:   {T_coolant_out:.2f} K")
print(f"Coolant-side wall temperature:{T_wall_coolant:.2f} K")
print(f"Bulk subcooling:              {bulk_subcooling:.2f} K")
print(f"Outlet subcooling:            {outlet_subcooling:.2f} K")
print(f"Wall subcooling:              {wall_subcooling:.2f} K")
print(f"Boiling predicted:            {boiling_predicted}")

assert wall_subcooling > 0.0

print("\nWall temperature is below saturation.")
print("Single-phase coolant assumption passes Model 0 screening.")

COOLANT BOILING SCREENING
-------------------------------------------
Coolant pressure:             15.00 bar(a)
Saturation temperature:       471.44 K
Coolant bulk temperature:     300.00 K
Coolant outlet temperature:   300.78 K
Coolant-side wall temperature:337.24 K
Bulk subcooling:              171.44 K
Outlet subcooling:            170.66 K
Wall subcooling:              134.19 K
Boiling predicted:            False

Wall temperature is below saturation.
Single-phase coolant assumption passes Model 0 screening.
